# QLoRA Fine-Tuning on Qwen3-8B

Training hyperparameters live in `config/config.yaml` (same file used by `src/fine_tune.py`).

In [ ]:
from pathlib import Path
import yaml

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

cfg_path = ROOT / "config" / "config.yaml"
with open(cfg_path, encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
print(yaml.dump(cfg, default_flow_style=False, sort_keys=False))


# Training Results

Summarized from the project training run (metrics printed below — no re-training in this cell).

In [ ]:
# Training run summary (reported)
print("Train loss:  2.905 → 1.014")
print("Val loss:    1.018")
print("Tok accuracy: 45% → 77%")
print("Trainable params (LoRA): ~43.6M")
print("Wall time: ~70 min")


# Model Loading and Test

Loads base **Qwen3-8B** + optional LoRA adapter from `checkpoints/lora_adapter` (same pattern as `scripts/generate_evaluation_results.py` / `src.chatbot.load_model`). GPU recommended.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.chatbot import ChatBot, load_model
from src.rag_pipeline import chunk_documents, create_vectorstore, load_corpus, load_vectorstore
from src.safety import StickySession
import yaml

cfg_path = ROOT / "config" / "config.yaml"
with open(cfg_path, encoding="utf-8") as f:
    full_cfg = yaml.safe_load(f) or {}
rag_cfg = full_cfg.get("rag") or {}
embed_model = rag_cfg.get("embedding_model", "sentence-transformers/all-MiniLM-L6-v2")
persist = ROOT / "chroma_db"
chunk_size = int(rag_cfg.get("chunk_size", 500))
overlap = int(rag_cfg.get("chunk_overlap", 50))

model, tokenizer = load_model(config_path=cfg_path)
vectorstore = None
if persist.exists() and any(persist.iterdir()):
    vectorstore = load_vectorstore(embedding_model=embed_model, persist_directory=persist)
else:
    docs = load_corpus(ROOT / "data" / "rag_corpus")
    if docs:
        chunks = chunk_documents(docs, chunk_size=chunk_size, chunk_overlap=overlap)
        vectorstore = create_vectorstore(chunks, embedding_model=embed_model, persist_directory=persist)

bot = ChatBot(model, tokenizer, vectorstore, config_path=cfg_path, max_new_tokens=256)
msg = "I've been feeling kind of empty lately"
reply = bot.respond(msg, session=StickySession(enable_sticky=False), history=[])
print("USER:", msg)
print("BOT:", reply)
